# v2 — 03: Pipeline & Cleaning
**Iteration 2** extends the v1 pipeline with:
- Draft capital features (pick, round, age at draft)
- Combine athleticism (40-yard dash, weight, vertical)
- Injury history (games missed, IR flag, career durability rate)
- College production (final season rec yards, TDs, dominator rate)
- **Rookies included** with NFL stat features set to 0

**Prerequisites:** Run `v1_01_data_gathering.ipynb` and `v2_01_data_gathering.ipynb` first.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.fetchers.nfl_fetcher import (
    fetch_seasonal_stats, fetch_rosters, fetch_snap_counts,
    fetch_draft_picks, fetch_combine_data, fetch_injuries,
)
from src.fetchers.sleeper_fetcher import fetch_players
from src.fetchers.college_fetcher import fetch_college_stats
from src.pipeline.cleaner import build_clean_dataset, read_from_db
from src.pipeline.features import build_model_dataset

pd.set_option('display.max_columns', 40)

## 1. Load All Data Sources (from cache)

In [2]:
# Set CFBD API key before fetching college stats
# Get your key at https://collegefootballdata.com/key
import os
os.environ["CFBD_API_KEY"] = "XwyjSybG0M2xEB6s1wvg42KTlPNyTXeMDEsdHXNM05fX0ycxaApPealTUNtKfFNN"  # <-- replace with your actual key

In [ ]:
# v1 sources
seasonal     = fetch_seasonal_stats()
rosters      = fetch_rosters()
snap_counts  = fetch_snap_counts()
sleeper      = fetch_players()

# v2 sources
draft_picks  = fetch_draft_picks()
combine_data = fetch_combine_data()
injuries     = fetch_injuries()
college_stats = fetch_college_stats(draft_seasons=list(range(2008, 2025)))

print(f'Seasonal:     {seasonal.shape}')
print(f'Rosters:      {rosters.shape}')
print(f'Snap counts:  {snap_counts.shape}')
print(f'Draft picks:  {draft_picks.shape}')
print(f'Combine:      {combine_data.shape}')
print(f'Injuries:     {injuries.shape}')
print(f'College:      {college_stats.shape}')

## 2. Run v2 Cleaning Pipeline

In [4]:
clean_df = build_clean_dataset(
    seasonal_df=seasonal,
    rosters_df=rosters,
    sleeper_df=sleeper,
    snap_df=snap_counts,
    draft_df=draft_picks,
    combine_df=combine_data,
    injury_df=injuries,
    college_df=college_stats,
)
print(f'\nClean dataset: {clean_df.shape}')
clean_df.head(3)

Cleaning seasonal stats...
Merging roster metadata...
Merging Sleeper metadata...
Merging snap pct...
Computing age at season...
Merging draft data...
Merging combine data...
Merging injury data...
Merging college stats...
College stats: 1571 player-seasons matched via name+draft_season
Loading to SQLite (nfl_stats)...
Loaded 5711 rows into table 'nfl_stats'
Clean dataset: 5711 rows, 92 columns

Clean dataset: (5711, 92)


,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,...,draft_round,age_at_draft,cfb_player_id,draft_season,is_undrafted,combine_forty,combine_weight,combine_height,combine_vertical,combine_bench,games_missed,ir_flag,college_rec_yards,college_rec_tds,college_targets,college_rush_yards,college_rush_tds,college_rush_atts,college_dominator_rate,college_years
0,00-0004541,2012,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.00000,0,0.000000,0.000000,0,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00-0006101,2012,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.00000,0,0.000000,0.000000,0,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00-0007091,2012,REG,138,221,1367.0,7,5.0,14.0,103.0,2,1,1707.0,557.0,71.0,-14.85631,1,6.316032,0.371353,13,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Inspect New Feature Coverage

In [5]:
new_v2_cols = [
    'draft_pick', 'draft_round', 'age_at_draft', 'is_undrafted',
    'combine_forty', 'combine_weight', 'combine_vertical',
    'games_missed', 'ir_flag',
    'college_rec_yards', 'college_dominator_rate',
]
print('Coverage of new v2 features:')
for col in new_v2_cols:
    if col in clean_df.columns:
        pct = clean_df[col].notna().mean()
        print(f'  {col:35s}: {pct:.1%} non-null')
    else:
        print(f'  {col:35s}: NOT IN DATASET')

Coverage of new v2 features:
  draft_pick                         : 53.9% non-null
  draft_round                        : 53.9% non-null
  age_at_draft                       : 53.9% non-null
  is_undrafted                       : 100.0% non-null
  combine_forty                      : 50.1% non-null
  combine_weight                     : 55.1% non-null
  combine_vertical                   : 47.0% non-null
  games_missed                       : 76.7% non-null
  ir_flag                            : 76.7% non-null
  college_rec_yards                  : 27.5% non-null
  college_dominator_rate             : 27.5% non-null


## 4. Feature Engineering (with Rookie Inclusion)

In [6]:
model_df = build_model_dataset(clean_df)
print(f'\nModel-ready dataset: {model_df.shape}')
model_df.head(3)

Adding rookie flag...
Adding lagged PPR features (rookies → 0)...
Adding lagged injury features...
Adding target variable...
Adding usage features...
Encoding position...
Selecting model features...
Model-ready dataset: 4058 rows, 45 features (547 rookies included)
Saving model_ready table to SQLite...
Saved 4058 rows to model_ready table

Model-ready dataset: (4058, 45)


,player_id,full_name,position,season,ppr_pts_next,is_rookie,age_at_season,ppr_pts_prev1,ppr_pts_prev2,fantasy_points_ppr,ppr_per_game,td_rate,target_share,air_yards_share,wopr,targets_per_game,carries_per_game,avg_snap_pct,receiving_epa,rushing_epa,...,draft_pick,draft_round,age_at_draft,is_undrafted,combine_forty,combine_weight,combine_height,combine_vertical,combine_bench,college_rec_yards,college_rec_tds,college_targets,college_rush_yards,college_rush_tds,college_dominator_rate,college_years,pos_QB,pos_RB,pos_WR,pos_TE
1,00-0006101,Tony Gonzalez,TE,2012,218.90,0,36.5,0.00,0.0,234.00,14.625,0.500,3.171097,3.236721,0.456716,7.75,0.000,NaN,61.123664,0.000000,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1
2,00-0007091,Matt Hasselbeck,QB,2012,16.94,0,36.9,0.00,0.0,76.48,9.560,0.875,0.000000,0.000000,0.000000,0.00,1.625,NaN,0.000000,-3.944240,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0
847,00-0007091,Matt Hasselbeck,QB,2014,91.10,0,38.9,76.48,0.0,16.94,4.235,0.500,0.000000,0.000000,0.000000,0.00,2.000,0.305,0.000000,-7.756981,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0


## 5. Validate Rookie Inclusion

In [7]:
model_df[['combine_forty', 'combine_weight', 'combine_height', 'combine_vertical',
       'combine_bench', 'college_rec_yards', 'college_rec_tds',
       'college_targets', 'college_rush_yards', 'college_rush_tds']].sum()

combine_forty           9639.21
combine_weight        499180.00
combine_height        167249.00
combine_vertical       68117.00
combine_bench          26011.00
college_rec_yards     594226.00
college_rec_tds         5004.00
college_targets        43815.00
college_rush_yards    434289.00
college_rush_tds        4661.00
dtype: float64

In [12]:
if 'is_rookie' in model_df.columns:
    rookies = model_df[model_df['is_rookie'] == 1]
    vets = model_df[model_df['is_rookie'] == 0]
    print(f'Rookies in model dataset: {len(rookies)}')
    print(f'Veterans in model dataset: {len(vets)}')
    print(f'\nRookie ppr_pts_prev1 (should all be 0): {rookies["ppr_pts_prev1"].unique()}')
    print(f'\nSample rookies:')
    cols = ['full_name', 'position', 'season', 'draft_pick', 'age_at_draft',
            'college_rec_yards', 'ppr_pts_next']
    print(rookies[[c for c in cols if c in rookies.columns]].head(10).to_string(index=False))

Rookies in model dataset: 547
Veterans in model dataset: 3511

Rookie ppr_pts_prev1 (should all be 0): [0.]

Sample rookies:
      full_name position  season  draft_pick  age_at_draft  college_rec_yards  ppr_pts_next
   Stephen Hill       WR    2012        43.0          21.0                NaN         62.20
 Alshon Jeffery       WR    2012        45.0          22.0                NaN        283.60
  Alfred Morris       RB    2012       173.0          23.0                NaN        178.30
   Ryan Broyles       WR    2012        54.0          24.0                NaN         16.50
  David Paulson       TE    2012       240.0          23.0                NaN         14.20
   David Wilson       RB    2012        32.0          21.0                NaN         19.40
LaMichael James       RB    2012        61.0          22.0                NaN          9.50
  Rueben Randle       WR    2012        63.0          21.0                NaN        138.10
 Russell Wilson       QB    2012        75.0   

## 6. Null Rates in Model Features

In [11]:
null_rates = model_df.isna().mean().sort_values(ascending=False)
print('Feature null rates (top 20):')
print(null_rates[null_rates > 0].head(20))

Feature null rates (top 20):
college_rec_yards         0.730655
college_dominator_rate    0.730655
college_rec_tds           0.730655
college_targets           0.730655
college_rush_tds          0.730655
college_rush_yards        0.730655
college_years             0.730655
combine_bench             0.634549
combine_vertical          0.516511
combine_forty             0.480286
age_at_draft              0.443568
draft_round               0.443568
draft_pick                0.443568
combine_height            0.435683
combine_weight            0.434943
avg_snap_pct              0.080089
dtype: float64
